# 01 - Data Exploration

Explore the Bangla toxic comment dataset: distributions, text lengths, word frequencies, and class balance.

In [ ]:
# Google Colab setup (does nothing when run locally).
# Clones the repo into Google Drive so results/ survive disconnects, then installs dependencies.
# Before the first run: Runtime > Change runtime type > T4 GPU, and upload train.csv (see below).
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Set_A'
    if not os.path.isdir(REPO_DIR):
        !git clone -b bangal_profanity https://github.com/mehediparvez/Set_A.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}/BanglaProfanityDetection/notebooks
    !pip install -q transformers datasets accelerate scikit-learn better-profanity groq google-generativeai pyyaml seaborn tabulate
    if not os.path.exists('../data/train.csv'):
        print(f'Missing dataset: upload train.csv to {REPO_DIR}/BanglaProfanityDetection/data/train.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys
sys.path.append('..')
from src.utils import load_config

sns.set_theme(style='whitegrid')
config = load_config('../configs/config.yaml')

## Load Dataset

In [ ]:
# Update this path to your actual dataset file
DATA_PATH = '../data/train.csv'
TEXT_COL = config['data']['text_column']
LABEL_COL = config['data']['label_column']

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## Class Distribution

In [ ]:
label_counts = df[LABEL_COL].value_counts()
label_names = {0: 'Non-Toxic', 1: 'Toxic'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(
    [label_names[i] for i in label_counts.index],
    label_counts.values,
    color=['#2ecc71', '#e74c3c']
)
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    label_counts.values,
    labels=[label_names[i] for i in label_counts.index],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c']
)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig('../plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClass balance ratio: {label_counts.min() / label_counts.max():.2f}')

## Text Length Analysis

In [ ]:
df['text_length'] = df[TEXT_COL].astype(str).apply(len)
df['word_count'] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#2ecc71', 'Non-Toxic'), (1, '#e74c3c', 'Toxic')]:
    subset = df[df[LABEL_COL] == label]
    axes[0].hist(subset['text_length'], bins=50, alpha=0.6, label=name, color=color)
    axes[1].hist(subset['word_count'], bins=50, alpha=0.6, label=name, color=color)

axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../plots/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby(LABEL_COL)[['text_length', 'word_count']].describe())

## Sample Texts

In [ ]:
print('=== Non-Toxic Samples ===')
for text in df[df[LABEL_COL] == 0][TEXT_COL].sample(5, random_state=42):
    print(f'  - {text[:100]}')

print('\n=== Toxic Samples ===')
for text in df[df[LABEL_COL] == 1][TEXT_COL].sample(5, random_state=42):
    print(f'  - {text[:100]}')

## Summary Statistics

In [ ]:
print(f'Total samples:    {len(df)}')
print(f'Non-toxic:        {(df[LABEL_COL] == 0).sum()}')
print(f'Toxic:            {(df[LABEL_COL] == 1).sum()}')
print(f'Avg text length:  {df["text_length"].mean():.0f} chars')
print(f'Avg word count:   {df["word_count"].mean():.0f} words')
print(f'Max text length:  {df["text_length"].max()} chars')
print(f'Null values:      {df[TEXT_COL].isna().sum()}')

In [ ]:
# Colab: make sure everything written to Google Drive is actually uploaded before the runtime ends
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All results saved to Google Drive.')